[◀ Notebook 02: Data Model](02_data_model.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 04: Simple Statements ▶](04_simple_statements.ipynb)

# Notebook 03: Expressions and Operations

An expression is a combination of values, variables, and operators that evaluates to a single value. This notebook details operator precedence, logical short-circuiting, and bitwise math.

Official Reference: [Python Language Reference - Expressions](https://docs.python.org/3/reference/expressions.html)

## 1. Operator Precedence & Associativity

When multiple operators appear in a single expression, Python evaluates them according to strict **operator precedence**.

| Precedence Level (Highest to Lowest) | Operator | Description |
|---|---|---|
| 1 | `(expressions...)` | Grouping |
| 2 | `x[index]`, `x[slice]`, `x(args...)`, `x.attribute` | Subscription, Slicing, Call, Attribute reference |
| 3 | `**` | Exponentiation (Right-associative!) |
| 4 | `+x`, `-x`, `~x` | Positive, Negative, Bitwise NOT |
| 5 | `*`, `@`, `/`, `//`, `%` | Multiplication, Matrix mult, Division, Floor division, Remainder |
| 6 | `+`, `-` | Addition, Subtraction |
| 7 | `<<`, `>>` | Bitwise Shifts |
| 8 | `&` | Bitwise AND |
| 9 | `^` | Bitwise XOR |
| 10 | `|` | Bitwise OR |
| 11 | `in`, `not in`, `is`, `is not`, `<`, `<=`, `>`, `>=`, `!=`, `==` | Comparisons and membership |
| 12 | `not x` | Boolean NOT |
| 13 | `and` | Boolean AND (Short-circuiting) |
| 14 | `or` | Boolean OR (Short-circuiting) |
| 15 | `lambda` | Lambda Expression |

### The Exponentiation GOTCHA
Unlike most binary operators which are left-associative, exponentiation (`**`) is **right-associative**. This means that:
`2 ** 3 ** 2` is evaluated as `2 ** (3 ** 2) = 2 ** 9 = 512`, NOT `(2 ** 3) ** 2 = 64`.

In [ ]:
# Exponentiation Right-Associativity Demo
print("2 ** 3 ** 2 =", 2 ** 3 ** 2) # 512
print("(2 ** 3) ** 2 =", (2 ** 3) ** 2) # 64

## 2. Comparison Chaining

Python supports comparison operator chaining (e.g., `a < b <= c`), which corresponds directly to mathematical intervals.

Under the hood, `a < b <= c` is equivalent to `a < b and b <= c`, with one key optimization:
**The middle operand `b` is evaluated exactly once.** This prevents performance penalties or side-effects if `b` is a function call.

In [ ]:
# Demonstrating evaluation efficiency in comparison chaining
def get_threshold():
    print("get_threshold() called!")
    return 5

# Evaluate 1 < get_threshold() <= 10
# Notice that "get_threshold() called!" is printed only once!
print("Chained comparison result:", 1 < get_threshold() <= 10)

## 3. Logical Operations & Short-circuiting

The boolean operators `and` and `or` perform **short-circuit evaluation**:

*   `x and y`: If `x` evaluates to a falsy value, it is returned immediately; `y` is never evaluated.
*   `x or y`: If `x` evaluates to a truthy value, it is returned immediately; `y` is never evaluated.

### Important: Values, Not Just Booleans
These operators do not restrict their return values to `True` or `False`. They return the **actual operand** evaluated last.

### The Falsy Short-Circuit Gotcha & Wrapping
A common design pattern is `A and B or C` to emulate a ternary operator (like `cond ? B : C`). However, if `B` is falsy (such as `0`, `0.0`, or `[]`), the expression evaluates as falsy, forcing `or C` to execute and return `C`. This breaks the intended behavior.

**The Wrapping Solution:** To handle falsy fallback returns without breaking evaluation, you can wrap targets inside single-element lists: `(cond and [B] or [C])[0]`. Because a non-empty list `[B]` is always truthy, the short-circuit expression will correctly choose `[B]` if `cond` is truthy, and `[C]` if `cond` is falsy. Indexing at `0` then extracts the actual value without risking unwanted fallbacks.

In [ ]:
# Return values of logical operators
print(0 or "default")      # "default" (0 is falsy, returns the second operand)
print("admin" and "guest") # "guest" ("admin" is truthy, returns the second operand)
print([] and 42)           # [] ([] is falsy, returns the first operand immediately)

# Short-circuit preventing errors
denom = 0
# This would raise ZeroDivisionError if evaluated, but short-circuiting saves us:
valid = (denom != 0) and (10 / denom > 2)
print("Is division safe?", valid)

# The Falsy Short-Circuit Gotcha & Wrapping Solution
# Emulating: condition ? B : C
# Mismatch case: B is falsy, e.g. B = 0.0
cond = True
fallback = -1.0
incorrect_result = cond and 0.0 or fallback
print("\nIncorrect result (without wrapping):", incorrect_result)

# Wrapping B and C in single-element lists ensures they evaluate as truthy
correct_wrapped = (cond and [0.0] or [fallback])[0]
print("Correct result (with list wrapping):", correct_wrapped)


## 4. Bitwise Operations

Bitwise operations treat integers as sequences of binary bits and perform calculations bit-by-bit:

*   `&` (AND), `|` (OR), `^` (XOR)
*   `~x`: Inverts all bits (equivalent to `-x - 1` due to two's complement representation)
*   `<<` (Left Shift), `>>` (Right Shift)

### Bitwise Shifting & Masking
-   **Shifting (`<<`, `>>`):** `x << shift` multiplies `x` by $2^{\text{shift}}$, while `x >> shift` floor-divides `x` by $2^{\text{shift}}$.
-   **Masking (`& 0xFF`):** Extracts the lowest 8 bits (one byte) of a value by clearing all higher-order bits. 
-   **Practical Example (Color Extraction):** A 24-bit RGB hex color (e.g. `0xFF5733`) encodes red, green, and blue values. We extract individual components by combining shifts and masks:
    -   Red: `(hex_color >> 16) & 0xFF` (shift right by 2 bytes, extract byte)
    -   Green: `(hex_color >> 8) & 0xFF` (shift right by 1 byte, extract byte)
    -   Blue: `hex_color & 0xFF` (extract lowest byte)

### Power-of-Two Bitwise Formula
To check if a positive integer `n` is a power of two, we can use the formula `n & (n - 1) == 0`:
-   **Mechanics:** In binary, a power of two `n` has exactly one bit set to `1` (e.g., `8 = 0b1000`). Subtracting `1` yields a binary sequence where all bits below that position are flipped to `1` and the power-of-two bit is `0` (e.g., `7 = 0b0111`).
-   **AND Result:** Performing `n & (n - 1)` clears the lowest set bit. Since a power of two has only one set bit, clearing it results in `0`. If `n` has multiple set bits, clearing the lowest set bit leaves the remaining bits intact, resulting in a non-zero value. (Note: The check must also verify `n > 0`).

In [ ]:
x = 0b1010  # 10 in decimal
y = 0b1100  # 12 in decimal

print("Bitwise Operators:")
print(f"x & y: {bin(x & y)}")  # AND: 0b1000 (8)
print(f"x | y: {bin(x | y)}")  # OR:  0b1110 (14)
print(f"x ^ y: {bin(x ^ y)}")  # XOR: 0b0110 (6)
print(f"~x:    {bin(~x)}")     # NOT: -11 (represented as -0b1011)

# Bitwise Shifting
val = 1
print("\nBitwise Shifting:")
print(f"val << 2 (multiply by 4): {val << 2}")
print(f"val >> 1 (floor-divide by 2): {val >> 1}")

# Bitwise Masking (Color Extraction)
# 24-bit hex color 0xFF5733 (Red=255, Green=87, Blue=51)
hex_color = 0xFF5733
r = (hex_color >> 16) & 0xFF
g = (hex_color >> 8) & 0xFF
b = hex_color & 0xFF
print(f"\nColor extraction for 0xFF5733: Red={r}, Green={g}, Blue={b}")

# Power-of-Two Bitwise Formula: n & (n - 1) == 0
def check_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0

print("\nPower of two checks:")
print(f"8 is power of two: {check_power_of_two(8)}")
print(f"10 is power of two: {check_power_of_two(10)}")


---

## Coding Challenges

Complete the logical and bitwise challenges below to practice expressions.

In [ ]:
# ==========================================
# Challenge 1: Logic Gate Emulation via Short-Circuiting
# ==========================================
# Instructions: Write a function `safe_divide_or_fallback(numerator, denominator, fallback)`
# that divides numerator by denominator, but uses short-circuiting to return `fallback` 
# if the denominator is 0. 
# 
# Requirements:
# - Do NOT use `if/else` statements, ternary operators, or `try/except` blocks.
# - You must use ONLY boolean short-circuit operators (`and`, `or`, and indexing if needed).
# - Crucially, your function MUST work even if the numerator evaluates to 0.0 or the fallback is 0.0.

def safe_divide_or_fallback(numerator: float, denominator: float, fallback: float) -> float:
    # TODO: Implement using short-circuit logical operators.
    pass

In [ ]:
# Test assertions for Challenge 1
try:
    assert safe_divide_or_fallback(10, 2, -1) == 5.0, "Failed standard division"
    assert safe_divide_or_fallback(10, 0, -1) == -1.0, "Failed zero division safety"
    assert safe_divide_or_fallback(0, 5, -9.9) == 0.0, "Failed for zero numerator"
    assert safe_divide_or_fallback(10, 0, 0.0) == 0.0, "Failed when fallback is 0.0"
    print("🎉 Challenge 1 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 1 Failed: {e}")

In [ ]:
# ==========================================
# Challenge 2: Power-of-Two Bitwise Checker
# ==========================================
# Instructions: Write a function `is_power_of_two(n)` that returns True if n is a positive 
# power of two (1, 2, 4, 8, 16...), and False otherwise.
# 
# Requirements:
# - Do NOT use loops (`for`, `while`).
# - Do NOT use string conversions or float-based operations.
# - Do NOT import the `math` module.
# - You must use bitwise operators (such as `&`, `|`, `^`, `<<`, `>>`, or subtraction tricks).

def is_power_of_two(n: int) -> bool:
    # TODO: Implement using bitwise arithmetic.
    pass

In [ ]:
# Test assertions for Challenge 2
try:
    assert is_power_of_two(1) == True, "1 is 2^0"
    assert is_power_of_two(2) == True, "2 is 2^1"
    assert is_power_of_two(1024) == True, "1024 is 2^10"
    assert is_power_of_two(10) == False, "10 is not a power of two"
    assert is_power_of_two(0) == False, "0 is not a power of two"
    assert is_power_of_two(-8) == False, "Negative numbers cannot be positive powers of two"
    print("🎉 Challenge 2 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 2 Failed: {e}")

In [ ]:
# ==========================================
# Challenge 3: Extract RGB Components via Bitwise Operators
# ==========================================
# Instructions: A 24-bit color is represented as a single integer (e.g. 0xFF5733 for red=255, green=87, blue=51).
# Write a function `extract_rgb(hex_color)` that takes a color integer and extracts the 
# individual red, green, and blue components using bitwise operators (`&`, shifts `<<`, `>>`).
# Return a tuple of ints: (red, green, blue) where each value is in the range [0, 255].

def extract_rgb(hex_color: int) -> tuple:
    # TODO: Implement using shifts and masks.
    pass

In [ ]:
# Test assertions for Challenge 3
try:
    assert extract_rgb(0xFF5733) == (255, 87, 51), "Failed for 0xFF5733"
    assert extract_rgb(0x00FF00) == (0, 255, 0), "Failed for pure Green"
    assert extract_rgb(0x010203) == (1, 2, 3), "Failed for low-value RGB"
    assert extract_rgb(0xFFFFFF) == (255, 255, 255), "Failed for White"
    print("🎉 Challenge 3 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 3 Failed: {e}")

[◀ Notebook 02: Data Model](02_data_model.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 04: Simple Statements ▶](04_simple_statements.ipynb)